In [ ]:
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

# ---------------------------------------------------------------- 1. modelling tables
claims = load_claims()
data = {}
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    F = build_features(line, "train", policies=p)
    data[line] = {"df": mt.merge(F, on="row_id", how="left", validate="one_to_one"), "loading": loading}
    print(f"{line}: table ready")

# ---------------------------------------------------------------- 2. settings to compare
FOLDS = [(2023, [2021, 2022]), (2024, [2021, 2022, 2023])]
CONFIGS = {"base (first, 1e-4, 10)": dict(drop="first", alpha=1e-4, n_bands=10)}
for a in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    for b in [10, 5]:
        CONFIGS[f"none, {a:g}, {b}"] = dict(drop=None, alpha=a, n_bands=b)

# ---------------------------------------------------------------- 3. run the grid
out = Path("../results"); out.mkdir(exist_ok=True)
res, t_all = [], time.time()
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in ("row_id", "year", "exposure_clean", "target", "weight")]
    for cname, kw in CONFIGS.items():
        t0 = time.time()
        for vy, tys in FOLDS:
            tr, va = df[df.year.isin(tys)], df[df.year == vy]
            glm = BandedTweedieGLM(**kw).fit(tr[feats], tr.target / tr.weight, tr.weight)
            pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            r = evaluate(va.target, pred, va.weight, name="", show=False)
            lf = r["lift"]["A/E"]
            res.append({"line": line, "config": cname, "year": vy, "D2": r["D2"], "A/E": r["A/E"],
                        "low": lf.iloc[0], "top": lf.iloc[-1], "top/bottom": r["top/bottom"]})
        pd.DataFrame(res).to_csv(out / "regularisation_grid.csv", index=False)   # save as we go
        print(f"{line} | {cname}: {time.time() - t0:.0f}s (total {(time.time() - t_all) / 60:.1f} min)")

# ---------------------------------------------------------------- 4. summary
R = pd.DataFrame(res)
pd.set_option("display.width", 220)
for line in LINES:
    s = R[R.line == line].pivot_table(index="config", columns="year", values=["D2", "A/E", "low", "top"])
    s.columns = [f"{m}_{y}" for m, y in s.columns]
    s["D2_avg"] = R[R.line == line].groupby("config").D2.mean()
    s["top/bottom_avg"] = R[R.line == line].groupby("config")["top/bottom"].mean()
    print(f"\n===== {line} =====")
    print(s.sort_values("D2_avg", ascending=False).round(3).to_string())

print("\nbest config per line (average D2 over 2023 and 2024):")
print(R.groupby(["line", "config"]).D2.mean().groupby(level=0).idxmax())

In [ ]:
# 07b Fine regularisation grid: drop=None, alpha 0.1-1, 10 vs 20 bands
# Purpose: find each line's best alpha between 0.1 and 1, and test whether more bands + ridge helps.
# Rule fixed before seeing results: highest average D2 over 2023/2024; if within 0.001 of the best,
# prefer 10 bands and the smaller alpha.
import sys, time, warnings
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

# 1. Modelling table per line (skipped if already built in this kernel)
if "data" not in globals():
    claims = load_claims()
    data = {}
    for line in LINES:
        p = load_policies(line, "train")
        mt, comp, loading = build_model_target(claims, p, line)
        F = build_features(line, "train", policies=p)
        df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
        assert len(df) == len(p)
        data[line] = {"df": df, "loading": loading}
        print(f"{line}: table ready")

CONFIGS = [(a, nb) for a in [0.1, 0.2, 0.3, 0.5, 1.0] for nb in [10, 20]]
NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")

# 2. Fit each config on both folds
rows, t_start = [], time.time()
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    for alpha, nb in CONFIGS:
        t0 = time.time()
        for vy, tys in FOLDS:
            tr, va = df[df.year.isin(tys)], df[df.year == vy]
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                glm = BandedTweedieGLM(alpha=alpha, n_bands=nb, drop=None).fit(
                    tr[feats], tr.target / tr.weight, tr.weight)
                pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            unknown = [str(x.message) for x in w if "unknown categories" in str(x.message)]
            r = evaluate(va.target, pred, va.weight, name=f"{line} {vy}", show=False)
            lf = r["lift"]["A/E"]
            rows.append({"line": line, "config": f"none, {alpha}, {nb}", "year": vy,
                         "D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1],
                         "top/bottom": r["top/bottom"], "unknown_cat_warning": "; ".join(unknown)})
        print(f"{line} | alpha {alpha}, bands {nb}: {time.time() - t0:.0f}s "
              f"(total {(time.time() - t_start) / 60:.1f} min)")

res = pd.DataFrame(rows)
Path("../results").mkdir(exist_ok=True)
res.to_csv("../results/regularisation_grid_fine.csv", index=False)

# 3. Per-line table, sorted by average D2
for line in LINES:
    t = res[res.line == line].pivot(index="config", columns="year",
                                    values=["A/E", "D2", "low", "top", "top/bottom"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.drop(columns=["top/bottom_2023", "top/bottom_2024"]).sort_values("D2_avg", ascending=False).round(4))

# 4. Pick per line with the pre-set rule
print("\nchosen config per line:")
for line in LINES:
    g = res[res.line == line].groupby("config")["D2"].mean()
    near = g[g >= g.max() - 0.001].index
    parsed = sorted(near, key=lambda c: (int(c.split(", ")[2]) != 10, float(c.split(", ")[1])))
    print(f"  {line}: best {g.idxmax()} (D2 {g.max():.4f}) -> chosen {parsed[0]} (D2 {g[parsed[0]]:.4f})")

# 5. Which column had categories unseen in training
w = res[res.unknown_cat_warning != ""][["line", "year", "unknown_cat_warning"]].drop_duplicates()
if len(w):
    cols = data["buildings"]["df"][[c for c in data["buildings"]["df"].columns if c not in NON_FEATURES]].columns
    print("\nunknown-category warnings:", w.drop_duplicates(["line", "year"]).to_string(index=False))
    print("buildings feature names by position (warning index refers to these):")
    print({i: c for i, c in enumerate(cols)})

In [ ]:
# 07c Balance correction, then compare the shortlisted configs
# Purpose: the log-link Tweedie GLM does not reproduce the training total, and ridge widens the gap,
# so configs differ in level as well as shape. Scale each model so that training predictions sum to
# training actuals (off-balance factor), then compare shape on equal footing.
# Rule (fixed before seeing results): highest average balanced D2; within 0.001, prefer the larger
# alpha (flatter = simpler), then 10 bands.
import time
import numpy as np
import pandas as pd
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

SHORTLIST = {
    "motor":     [(0.2, 10), (0.3, 10), (0.5, 10), (0.3, 20)],
    "buildings": [(0.3, 20), (0.5, 20), (1.0, 20), (0.5, 10), (1.0, 10)],
    "contents":  [(0.1, 20), (0.2, 20), (0.3, 20), (0.3, 10)],
}
NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")

rows = []
for line, configs in SHORTLIST.items():
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    for alpha, nb in configs:
        t0 = time.time()
        for vy, tys in FOLDS:
            tr, va = df[df.year.isin(tys)], df[df.year == vy]
            glm = BandedTweedieGLM(alpha=alpha, n_bands=nb, drop=None).fit(
                tr[feats], tr.target / tr.weight, tr.weight)
            p_tr = glm.predict_rate(tr[feats]) * tr.weight.to_numpy()
            k = tr.target.sum() / p_tr.sum()                 # off-balance factor from training years only
            p_va = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            for label, pred in [("raw", p_va), ("balanced", p_va * k)]:
                r = evaluate(va.target, pred, va.weight, show=False)
                lf = r["lift"]["A/E"]
                rows.append({"line": line, "alpha": alpha, "bands": nb, "year": vy, "version": label,
                             "k": k, "D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]})
        print(f"{line} | alpha {alpha}, bands {nb}: {time.time() - t0:.0f}s")

bal = pd.DataFrame(rows)
bal.to_csv("../results/regularisation_balanced.csv", index=False)

for line in SHORTLIST:
    t = bal[bal.line == line].pivot_table(index=["alpha", "bands", "version"], columns="year",
                                          values=["k", "D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\nchosen config per line (balanced):")
for line in SHORTLIST:
    g = bal[(bal.line == line) & (bal.version == "balanced")].groupby(["alpha", "bands"])["D2"].mean()
    near = g[g >= g.max() - 0.001]
    pick = sorted(near.index, key=lambda ab: (-ab[0], ab[1] != 10))[0]
    print(f"  {line}: best {g.idxmax()} (D2 {g.max():.4f}) -> chosen alpha {pick[0]}, bands {pick[1]} "
          f"(D2 {g[pick]:.4f})")

In [ ]:
# 08 Model v2: regularised, balanced GLM per line -> first Kaggle submission
# Purpose: refit the chosen config on 2021-2024, predict 2025, run pre-submission checks, write the file.
import sys
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import DATA_DIR, LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM

CONFIG = {  # chosen in 07c (see DECISIONS.md)
    "motor":     dict(alpha=0.5, n_bands=10, drop=None, balance=True),
    "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
    "contents":  dict(alpha=0.3, n_bands=20, drop=None, balance=True),
}
NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")

claims = load_claims()
subs, models, checks = [], {}, []
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    df = mt.merge(build_features(line, "train", policies=p), on="row_id", how="left", validate="one_to_one")
    feats = [c for c in df.columns if c not in NON_FEATURES]

    glm = BandedTweedieGLM(**CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)
    models[line] = glm

    pt = load_policies(line, "test")
    Ft = build_features(line, "test", policies=pt)
    assert list(Ft.row_id) == list(pt.row_id)
    rate = glm.predict_rate(Ft[feats])
    pred = np.maximum((rate + loading) * pt.exposure_clean.to_numpy(), 0)
    subs.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred}))

    # level check: test predicted rate (exposure-weighted) vs training level, both incl. loading
    train_level = df.target.sum() / df.weight.sum() + loading
    test_level = pred.sum() / pt.exposure_clean.sum()
    checks.append({"line": line, "k": glm.k_, "loading": loading, "train_level": train_level,
                   "test_level": test_level, "test/train": test_level / train_level,
                   "test_rows": len(pt), "total_pred": pred.sum()})
    print(f"{line}: fitted, k = {glm.k_:.4f}")

chk = pd.DataFrame(checks).set_index("line")
print("\nlevel check (test/train should be close to 1 unless the 2025 mix differs):")
print(chk.round(3))

# pre-submission checks
sub = pd.concat(subs, ignore_index=True)
sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
assert len(sub) == len(sample), "row count differs from sample_submission"
assert set(sub.row_id) == set(sample.row_id), "row_ids differ from sample_submission"
assert sub.row_id.is_unique, "duplicate row_id"
assert sub.prediction.notna().all(), "missing predictions"
assert (sub.prediction >= 0).all(), "negative predictions"
sub = sample[["row_id"]].merge(sub, on="row_id", how="left")      # same order as sample_submission

# change against the baseline file, if it exists
base_path = Path("../submissions/glm_baseline.csv")
if base_path.exists():
    b = pd.read_csv(base_path).merge(sub, on="row_id", suffixes=("_base", "_v2"))
    r = b.prediction_v2 / b.prediction_base.replace(0, np.nan)
    print("\nv2 / baseline per row:", r.describe(percentiles=[.01, .5, .99]).round(3).to_dict())
    print(f"rows changed by more than 2x either way: {((r > 2) | (r < 0.5)).mean():.2%}")

out = Path("../submissions"); out.mkdir(exist_ok=True)
sub.to_csv(out / "glm_v2.csv", index=False)
print(f"\nwrote {len(sub):,} rows to submissions/glm_v2.csv")
print(sub.prediction.describe().round(1))

In [ ]:
# 08b Why is the motor 2025 predicted level 7% below training?
# Purpose: (1) predicted level by year, to see if 2025 continues a mix trend or breaks from it;
# (2) split the 2024 -> 2025 change in predicted level by feature (log-relativity contributions).
line = "motor"
glm = models[line]
p = load_policies(line, "train")
mt, comp, loading = build_model_target(claims, p, line)
df = mt.merge(build_features(line, "train", policies=p), on="row_id", how="left", validate="one_to_one")
feats = [c for c in df.columns if c not in NON_FEATURES]
pt = load_policies(line, "test")
Ft = build_features(line, "test", policies=pt)

X = pd.concat([df[feats].assign(_year=df.year.values, _exp=df.exposure_clean.values),
               Ft[feats].assign(_year=2025, _exp=pt.exposure_clean.values)], ignore_index=True)

# (1) exposure-weighted predicted rate by year (same model, so only the mix differs)
X["_rate"] = glm.predict_rate(X[feats])
lvl = X.groupby("_year").apply(lambda g: np.average(g._rate, weights=g._exp))
print("predicted rate by year (motor, before loading):")
print((lvl / lvl.loc[2024]).round(3).rename("relative to 2024").to_string())

# (2) contribution of each feature to the 2024 -> 2025 change
Z = glm.enc_.transform(glm._band(X[feats])).tocsc()
names = glm.enc_.get_feature_names_out()
fnames = sorted(glm.enc_.feature_names_in_, key=len, reverse=True)
owner = np.array([next(f for f in fnames if n.startswith(f + "_")) for n in names])
coef = glm.model_.coef_
w24 = np.where(X._year == 2024, X._exp, 0); w24 = w24 / w24.sum()
w25 = np.where(X._year == 2025, X._exp, 0); w25 = w25 / w25.sum()
rows = []
for f in glm.enc_.feature_names_in_:
    idx = np.where(owner == f)[0]
    contrib = Z[:, idx] @ coef[idx]                  # each row's log-relativity from this feature
    rows.append({"feature": f, "change_%": (np.exp(w25 @ contrib - w24 @ contrib) - 1) * 100})
ch = pd.DataFrame(rows).sort_values("change_%")
print("\nchange in average log-relativity, 2025 vs 2024 (%), largest effects:")
print(pd.concat([ch.head(6), ch.tail(3)]).round(2).to_string(index=False))
print(f"\nsum of feature effects: {(np.exp(np.log1p(ch['change_%'] / 100).sum()) - 1) * 100:.1f}%  "
      f"(approximately equals the 2025 vs 2024 level change above)")

# (3) for the top 3 features: distribution in 2024 vs 2025
for f in ch.head(3).feature:
    s24 = df.loc[df.year == 2024, f]; s25 = Ft[f]
    if pd.api.types.is_numeric_dtype(s24) and s24.nunique() > 10:
        print(f"\n{f}: mean 2024 {s24.mean():.2f} | 2025 {s25.mean():.2f}; "
              f"missing 2024 {s24.isna().mean():.1%} | 2025 {s25.isna().mean():.1%}")
    else:
        t = pd.concat([s24.value_counts(normalize=True, dropna=False).rename("2024"),
                       s25.value_counts(normalize=True, dropna=False).rename("2025")], axis=1).fillna(0)
        print(f"\n{f}:"); print(t.round(3).head(10).to_string())

In [1]:
# 08c Version B: excess also deducted on liability perils (motor only)
# Purpose: test the other reading of "net of the policy excess" on the leaderboard.
# Same model config as v2; only the motor target changes. Buildings/contents rows are copied from v2.
import sys
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
from src.target import DATA_DIR, load_policies, load_claims, build_model_target
from src.features import build_features
from src.glm import BandedTweedieGLM

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
claims = load_claims()
line = "motor"
p = load_policies(line, "train")
mt, comp, loading = build_model_target(claims, p, line, excess_on_liability=True)
df = mt.merge(build_features(line, "train", policies=p), on="row_id", how="left", validate="one_to_one")
feats = [c for c in df.columns if c not in NON_FEATURES]
glm = BandedTweedieGLM(alpha=0.5, n_bands=10, drop=None, balance=True).fit(
    df[feats], df.target / df.weight, df.weight)

pt = load_policies(line, "test")
Ft = build_features(line, "test", policies=pt)
pred_b = np.maximum((glm.predict_rate(Ft[feats]) + loading) * pt.exposure_clean.to_numpy(), 0)
motor_b = pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_b})

v2 = pd.read_csv("../submissions/glm_v2.csv")
sub = v2.copy()
sub = sub.set_index("row_id")
sub.loc[motor_b.row_id, "prediction"] = motor_b.prediction.values
sub = sub.reset_index()

# checks: same rows/order as v2; only motor rows changed; motor total should fall a few %
assert len(sub) == len(v2) and (sub.row_id == v2.row_id).all()
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
is_motor = sub.row_id.isin(motor_b.row_id)
assert np.allclose(sub.prediction[~is_motor], v2.prediction[~is_motor])
ratio = sub.prediction[is_motor].sum() / v2.prediction[is_motor].sum()
print(f"motor: k = {glm.k_:.4f}, loading {loading:.1f}; motor total B / v2 = {ratio:.3f} "
      f"(expected roughly 0.95: excess on liability lowered the motor total by about 4.6%)")

sub.to_csv("../submissions/glm_v2_excess_liability.csv", index=False)
print(f"wrote {len(sub):,} rows to submissions/glm_v2_excess_liability.csv")

motor: k = 1.0197, loading 16.0; motor total B / v2 = 0.957 (expected roughly 0.95: excess on liability lowered the motor total by about 4.6%)
wrote 626,639 rows to submissions/glm_v2_excess_liability.csv
